In [2]:
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from jpeg_compression_detection import (
    analyze_dct_spectrum,
    detect_jpeg_blockiness,
    detect_jpeg_compression,
    estimate_jpeg_quality,
)
from jpeg_compression_detection.paths import (
    DATA_DIR,
    GENERATED_DATA_DIR,
    display_path,
)


# Convert one image to another file format


In [3]:
source_image_path = DATA_DIR / "contest-images" / "00732_960620_fa.ppm"
single_conversion_dir = GENERATED_DATA_DIR / "single-file-conversion"

if not source_image_path.is_file():
    print(f"Single-file conversion skipped; input is missing: {display_path(source_image_path)}")
else:
    single_conversion_dir.mkdir(parents=True, exist_ok=True)
    output_png_path = single_conversion_dir / f"{source_image_path.name}.png"
    with Image.open(source_image_path) as image:
        image.save(output_png_path)
    print(f"Saved: {display_path(output_png_path)}")


# Convert a series of camera-RAW files


In [ ]:
raw_camera_dir = DATA_DIR / "contest-images"
raw_conversion_dir = GENERATED_DATA_DIR / "raw-conversions"
raw_extensions = {".dng", ".nef", ".cr2", ".arw", ".raw", ".rw2", ".orf", ".pef"}
RUN_RAW_CONVERSION = False

if not RUN_RAW_CONVERSION:
    print("RAW conversion skipped. Set RUN_RAW_CONVERSION = True to enable it.")
elif not raw_camera_dir.is_dir():
    print(f"RAW conversion skipped; input directory is missing: {display_path(raw_camera_dir)}")
else:
    try:
        import imageio.v3 as iio
        import rawpy
    except ImportError as exc:
        print(f"RAW conversion dependency is unavailable: {exc}")
    else:
        raw_conversion_dir.mkdir(parents=True, exist_ok=True)
        for raw_path in sorted(raw_camera_dir.iterdir()):
            if not raw_path.is_file() or raw_path.suffix.lower() not in raw_extensions:
                continue
            try:
                with rawpy.imread(str(raw_path)) as raw_image:
                    rgb = raw_image.postprocess()
                output_png = raw_conversion_dir / f"{raw_path.name}.png"
                iio.imwrite(output_png, rgb)
                print(f"Saved: {output_png}")
            except Exception as exc:
                print(f"Unable to convert {display_path(raw_path)}: {exc}")


# JPEG and JPEG 2000 compression


In [5]:
compression_source_path = DATA_DIR / "test-images" / "00085_931230_pl.ppm"
format_conversion_dir = GENERATED_DATA_DIR / "format-conversions"

if not compression_source_path.is_file():
    print(f"Compression example skipped; input is missing: {display_path(compression_source_path)}")
else:
    format_conversion_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(compression_source_path) as image:
        image.convert("RGB").save(format_conversion_dir / f"{compression_source_path.name}.jpg", "JPEG", quality=85)
        image.convert("RGB").save(format_conversion_dir / f"{compression_source_path.name}-100-percent.jpg", "JPEG", quality=100)
        try:
            image.save(format_conversion_dir / f"{compression_source_path.name}.jp2", "JPEG2000", quality_mode="rates", quality_layers=[0.3])
        except (OSError, ValueError) as exc:
            print(f"JPEG 2000 output skipped because Pillow lacks the required codec: {exc}")


# Explore blockiness with a Laplacian edge detector


In [ ]:
laplacian_image_path = DATA_DIR / "test-images" / "00085_931230_pl.ppm"
image = cv2.imread(str(laplacian_image_path)) if laplacian_image_path.is_file() else None

if image is None:
    print(f"Laplacian example skipped; image is missing or unreadable: {display_path(laplacian_image_path)}")
else:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original image")
    plt.axis("off")

    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian_abs = np.uint8(np.absolute(laplacian))
    _, thresholded = cv2.threshold(laplacian_abs, 50, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresholded, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    marked_image = image.copy()
    cv2.drawContours(marked_image, contours, -1, (0, 0, 255), 1)
    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(marked_image, cv2.COLOR_BGR2RGB))
    plt.title("Detected high-contrast contours")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# Define visualization helpers for the compression indicators


In [20]:
def visualize_blockiness(image):
    """Return a heat map of differences across 8 x 8 block boundaries."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) > 2 else image
    blockiness_map = np.zeros_like(gray, dtype=float)
    for row in range(7, gray.shape[0], 8):
        if row + 1 < gray.shape[0]:
            blockiness_map[row, :] = np.abs(
                np.float32(gray[row, :]) - np.float32(gray[row + 1, :])
            )
    for column in range(7, gray.shape[1], 8):
        if column + 1 < gray.shape[1]:
            blockiness_map[:, column] += np.abs(
                np.float32(gray[:, column]) - np.float32(gray[:, column + 1])
            )
    normalized = cv2.normalize(blockiness_map, None, 0, 255, cv2.NORM_MINMAX)
    return normalized.astype(np.uint8)


def inspect_jpeg_compression(image_path):
    """Display the image, block grid, heat map, and repository indicators."""
    image_path = Path(image_path)
    image = cv2.imread(str(image_path)) if image_path.is_file() else None
    if image is None:
        print(f"Inspection skipped; image is missing or unreadable: {display_path(image_path)}")
        return None

    indicators = detect_jpeg_compression(image_path)
    if not isinstance(indicators, dict):
        print(f"Unable to calculate indicators: {indicators}")
        return None

    ratios = np.array([indicators[f"blockiness_{index}"] for index in range(8)])
    is_jpeg = indicators["blockiness_max"] > 1.1
    heat_map = visualize_blockiness(image)

    figure = plt.figure(figsize=(15, 10))
    plt.subplot(2, 2, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(2, 2, 2)
    plt.imshow(heat_map, cmap="hot")
    plt.title(f"Blockiness map (max: {indicators['blockiness_max']:.2f})")
    plt.axis("off")

    grid_image = image.copy()
    height, width = image.shape[:2]
    for column in range(0, width, 8):
        cv2.line(grid_image, (column, 0), (column, height), (0, 255, 0), 1)
    for row in range(0, height, 8):
        cv2.line(grid_image, (0, row), (width, row), (0, 255, 0), 1)
    plt.subplot(2, 2, 3)
    plt.imshow(cv2.cvtColor(grid_image, cv2.COLOR_BGR2RGB))
    plt.title("8 x 8 block grid")
    plt.axis("off")

    lines = [
        "Analysis results:",
        f"- JPEG blockiness threshold exceeded: {'YES' if is_jpeg else 'NO'}",
        f"- Blockiness ratios: {np.array2string(ratios, precision=3)}",
        f"- Mean DCT energy: {indicators['energy_mean']:.2f}",
        f"- DCT-energy standard deviation: {indicators['energy_std']:.2f}",
        f"- Estimated JPEG quality: {indicators['estimated_quality']}%",
        f"- PSNR: {indicators['psnr']:.2f} dB",
    ]
    plt.subplot(2, 2, 4)
    plt.axis("off")
    plt.text(0.05, 0.5, "\n".join(lines), fontsize=9)
    plt.tight_layout()
    plt.show()
    return indicators


### Generate a JPEG quality sweep from PPM/PNG inputs


In [ ]:
ppm_source_dir = DATA_DIR / "ppm-copy"
ppm_sweep_dir = GENERATED_DATA_DIR / "ppm-quality-sweep"
output_qualities = [10, 20, 30, 40, 50, 60, 70, 75, 78, 80, 82, 84, 86, 88, 89, 90, 91, 92, 93, 94, 95]
RUN_PPM_QUALITY_SWEEP = False

if not RUN_PPM_QUALITY_SWEEP:
    print("PPM quality sweep skipped. Set RUN_PPM_QUALITY_SWEEP = True to enable it.")
elif not ppm_source_dir.is_dir():
    print(f"PPM quality sweep skipped; source directory is missing: {display_path(ppm_source_dir)}")
else:
    ppm_sweep_dir.mkdir(parents=True, exist_ok=True)
    input_images = sorted(path for path in ppm_source_dir.iterdir() if path.suffix.lower() in {".png", ".ppm"})
    for image_path in input_images:
        try:
            with Image.open(image_path) as image:
                rgb_image = image.convert("RGB")
                for quality in output_qualities:
                    output_path = ppm_sweep_dir / f"{image_path.name}_q{quality}.jpg"
                    rgb_image.save(output_path, "JPEG", quality=quality)
                    print(f"Saved: {display_path(output_path)}")
        except Exception as exc:
            print(f"Unable to compress {display_path(image_path)}: {exc}")
    print(f"Generated at most {len(input_images) * len(output_qualities)} JPEG files")


In [30]:
# Optional interactive inspection of every test image.
# test_images_dir = DATA_DIR / "test-images"
# if test_images_dir.is_dir():
#     for image_path in sorted(test_images_dir.iterdir()):
#         inspect_jpeg_compression(image_path)


## Small, targeted tests


In [29]:
# Examples from an optional synthetic-face data set.
# synthetic_faces_dir = DATA_DIR / "thispersondoesnotexist-10k"
# inspect_jpeg_compression(synthetic_faces_dir / "0.jpg")
# inspect_jpeg_compression(synthetic_faces_dir / "1.jfif")
#
# jfif_dir = DATA_DIR / "jfif"
# inspect_jpeg_compression(jfif_dir / "person_0000.jfif")
# inspect_jpeg_compression(jfif_dir / "person_0001.jfif")
# inspect_jpeg_compression(DATA_DIR / "test-images" / "00-swapped-woman.jpg")


# Optional data acquisition for indicator tests


In [ ]:
DOWNLOAD_SYNTHETIC_FACES = False

if not DOWNLOAD_SYNTHETIC_FACES:
    print("Synthetic-face download skipped. Set DOWNLOAD_SYNTHETIC_FACES = True to enable it.")
else:
    import random
    import time

    import requests
    from tqdm.notebook import tqdm

    download_dir = GENERATED_DATA_DIR / "synthetic-face-downloads"
    download_dir.mkdir(parents=True, exist_ok=True)
    for index in tqdm(range(10_000)):
        try:
            response = requests.get(
                "https://thispersondoesnotexist.com/",
                headers={"User-Agent": "research-reproduction/1.0"},
                timeout=30,
            )
            response.raise_for_status()
            (download_dir / f"person_{index + 1001:05d}.jfif").write_bytes(response.content)
            time.sleep(random.uniform(0.1, 0.3))
        except requests.RequestException as exc:
            print(f"Download failed for image {index}: {exc}")
    print("Download complete")


# Repeated JPEG compression experiments


## Convert baseline PPM images to PNG


In [11]:
RUN_PPM_TO_PNG_CONVERSION = False
multiple_compression_raw_dir = DATA_DIR / "multiple-compression"
multiple_compression_png_dir = GENERATED_DATA_DIR / "multiple-compression-png"

if not RUN_PPM_TO_PNG_CONVERSION:
    print("PPM-to-PNG conversion skipped. Set RUN_PPM_TO_PNG_CONVERSION = True to enable it.")
elif not multiple_compression_raw_dir.is_dir():
    print(f"PPM-to-PNG conversion skipped; source directory is missing: {display_path(multiple_compression_raw_dir)}")
else:
    multiple_compression_png_dir.mkdir(parents=True, exist_ok=True)
    for ppm_path in sorted(multiple_compression_raw_dir.glob("*.ppm")):
        try:
            with Image.open(ppm_path) as image:
                output_path = multiple_compression_png_dir / f"{ppm_path.stem}.png"
                image.save(output_path)
                print(f"Saved: {display_path(output_path)}")
        except Exception as exc:
            print(f"Unable to convert {display_path(ppm_path)}: {exc}")


## Optional cleanup of generated files


In [ ]:
RUN_GENERATED_DATA_CLEANUP = False
cleanup_dir = GENERATED_DATA_DIR / "multiple-compression"

if not RUN_GENERATED_DATA_CLEANUP:
    print("Cleanup skipped. Set RUN_GENERATED_DATA_CLEANUP = True to enable it.")
elif cleanup_dir.resolve().parent != GENERATED_DATA_DIR.resolve():
    raise RuntimeError(f"Refusing to clean a directory outside GENERATED_DATA_DIR: {display_path(cleanup_dir)}")
elif not cleanup_dir.is_dir():
    print(f"Cleanup skipped; directory is missing: {display_path(cleanup_dir)}")
else:
    for path in cleanup_dir.iterdir():
        if path.is_file() and ".q" in path.name.lower():
            path.unlink()
            print(f"Removed generated file: {path.name}")
    print("Cleanup complete")


## Three compression rounds with independently varied quality


In [ ]:
def generate_recompression_dataset(raw_dir, work_dir, quality_levels, enabled=False):
    """Stage raw inputs and create up to three JPEG generations in a generated-data directory."""
    raw_dir = Path(raw_dir)
    work_dir = Path(work_dir)
    if not enabled:
        print(f"Generation skipped for {work_dir.name}; enable its RUN_* flag to execute it.")
        return
    if not raw_dir.is_dir():
        print(f"Generation skipped; raw input directory is missing: {display_path(raw_dir)}")
        return
    if work_dir.resolve().parent != GENERATED_DATA_DIR.resolve():
        raise RuntimeError(f"Refusing to modify a directory outside GENERATED_DATA_DIR: {display_path(work_dir)}")

    work_dir.mkdir(parents=True, exist_ok=True)
    for source_path in raw_dir.iterdir():
        if source_path.is_file() and source_path.suffix.lower() in {".png", ".ppm"}:
            shutil.copy2(source_path, work_dir / source_path.name)
    for path in work_dir.iterdir():
        if path.is_file() and path.suffix.lower() not in {".png", ".ppm"}:
            path.unlink()

    source_images = sorted(
        path for path in work_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".png", ".ppm"}
    )
    for source_image_path in source_images:
        with Image.open(source_image_path) as image:
            rgb_image = image.convert("RGB")
            for quality in quality_levels:
                first_path = work_dir / f"{source_image_path.stem}_q{quality}.jpg"
                rgb_image.save(first_path, "JPEG", quality=quality)

    first_generation = sorted(work_dir.glob("*.jpg"))
    for first_path in first_generation:
        with Image.open(first_path) as first_image:
            rgb_first = first_image.convert("RGB")
            for second_quality in quality_levels:
                second_path = work_dir / f"{first_path.name}.q{second_quality}.jpg"
                rgb_first.save(second_path, "JPEG", quality=second_quality)
                with Image.open(second_path) as second_image:
                    for third_quality in quality_levels:
                        third_path = work_dir / f"{first_path.name}.q{second_quality}.q{third_quality}.jpg"
                        second_image.save(third_path, "JPEG", quality=third_quality)
    print(f"Generated repeated-compression data in {display_path(work_dir)}")


RUN_DIFFERENT_QUALITY_EXPERIMENT = False
different_quality_dir = GENERATED_DATA_DIR / "multiple-compression-different-quality"
generate_recompression_dataset(
    DATA_DIR / "multiple-compression",
    different_quality_dir,
    [10, 30, 50, 70, 85, 95, 100],
    enabled=RUN_DIFFERENT_QUALITY_EXPERIMENT,
)


## Three compression rounds with the standard quality grid


In [ ]:
RUN_STANDARD_QUALITY_EXPERIMENT = False
standard_quality_dir = GENERATED_DATA_DIR / "multiple-compression-standard-quality"
generate_recompression_dataset(
    DATA_DIR / "multiple-compression",
    standard_quality_dir,
    [10, 30, 50, 65, 75, 85, 95],
    enabled=RUN_STANDARD_QUALITY_EXPERIMENT,
)


## Calculate indicators for the repeated-compression data


In [ ]:
RUN_MULTIPLE_COMPRESSION_ANALYSIS = False
analysis_results_path = GENERATED_DATA_DIR / "results_multiple_compression_different_quality.csv"
results = []

if not RUN_MULTIPLE_COMPRESSION_ANALYSIS:
    print("Indicator analysis skipped. Set RUN_MULTIPLE_COMPRESSION_ANALYSIS = True to enable it.")
elif not different_quality_dir.is_dir():
    print(f"Indicator analysis skipped; input directory is missing: {display_path(different_quality_dir)}")
else:
    image_paths = sorted(
        path for path in different_quality_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".ppm"}
    )
    for image_path in image_paths:
        indicators = detect_jpeg_compression(image_path)
        if isinstance(indicators, dict):
            results.append({"filename": image_path.name, **indicators})
        else:
            print(f"Unable to analyze {image_path.name}: {indicators}")

    if results:
        analysis_results_path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(results).to_csv(analysis_results_path, index=False)
        print(f"Saved {len(results)} rows to {display_path(analysis_results_path)}")


## Present summary statistics


In [ ]:
results_path = GENERATED_DATA_DIR / "results_multiple_compression_different_quality.csv"
results_df = pd.read_csv(results_path) if results_path.is_file() else pd.DataFrame()

if results_df.empty:
    print(f"Summary plots skipped; result table is missing or empty: {display_path(results_path)}")
else:
    def get_compression_count(filename):
        q_count = filename.count(".q")
        if filename.endswith((".png", ".ppm")):
            return 0
        return 1 if q_count == 0 else q_count + 1

    results_df["compression_count"] = results_df["filename"].apply(get_compression_count)
    font_size = 16
    colors = ["#BD4E59", "#35A14D", "#3784BE", "#B4B442", "#B17633"]
    alpha = 0.8
    compression_levels = sorted(results_df["compression_count"].unique())
    figure, axes = plt.subplots(2, 2, figsize=(15, 10))

    blockiness_data = [results_df.loc[results_df["compression_count"] == level, "blockiness_max"].values for level in compression_levels]
    axes[0, 0].hist(blockiness_data, bins=30, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True, alpha=alpha)
    axes[0, 0].set_xlabel("Maximum blockiness ratio", fontsize=font_size)
    axes[0, 0].set_ylabel("Number of images", fontsize=font_size)
    axes[0, 0].set_title("Maximum blockiness distribution", fontsize=font_size)
    axes[0, 0].set_yscale("log")
    axes[0, 0].legend(fontsize=font_size)

    groups = results_df.groupby("compression_count")["blockiness_max"].apply(list)
    boxplot = axes[0, 1].boxplot(groups.values, labels=groups.index, patch_artist=True)
    for patch, color in zip(boxplot["boxes"], colors[: len(groups)]):
        patch.set_facecolor(color)
        patch.set_alpha(alpha)
    axes[0, 1].set_xlabel("Number of compressions", fontsize=font_size)
    axes[0, 1].set_ylabel("Maximum blockiness ratio", fontsize=font_size)
    axes[0, 1].set_title("Blockiness vs number of compressions", fontsize=font_size)

    energy_data = [results_df.loc[results_df["compression_count"] == level, "energy_mean"].values for level in compression_levels]
    axes[1, 0].hist(energy_data, bins=30, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True, alpha=alpha)
    axes[1, 0].set_xlabel("Mean DCT energy", fontsize=font_size)
    axes[1, 0].set_ylabel("Number of images", fontsize=font_size)
    axes[1, 0].set_title("DCT-energy distribution", fontsize=font_size)
    axes[1, 0].set_yscale("log")

    scatter = axes[1, 1].scatter(results_df["energy_mean"], results_df["energy_std"], c=results_df["compression_count"], cmap="viridis", alpha=alpha, s=20)
    axes[1, 1].set_xlabel("Mean DCT energy", fontsize=font_size)
    axes[1, 1].set_ylabel("DCT-energy standard deviation", fontsize=font_size)
    axes[1, 1].set_title("Mean vs standard deviation of DCT energy", fontsize=font_size)
    plt.colorbar(scatter, ax=axes[1, 1], label="Number of compressions")
    plt.tight_layout()
    plt.show()


## Compare JPEG-quality estimation methods


In [ ]:
quality_columns = ["quality_recompression", "quality_dct", "quality_histogram", "estimated_quality"]

if results_df.empty:
    print("Quality-method comparison skipped because no result table is available.")
elif not set(quality_columns).issubset(results_df.columns):
    missing_columns = sorted(set(quality_columns) - set(results_df.columns))
    print(f"Quality-method comparison skipped; missing columns: {missing_columns}")
else:
    figure, axes = plt.subplots(2, 2, figsize=(16, 12))
    method_specs = [
        ("quality_recompression", "Recompression", "viridis"),
        ("quality_dct", "DCT", "plasma"),
        ("quality_histogram", "Histogram", "coolwarm"),
    ]
    for axis, (column, label, color_map) in zip(axes.flat[:3], method_specs):
        scatter = axis.scatter(results_df[column], results_df["estimated_quality"], c=results_df["compression_count"], cmap=color_map, alpha=0.6, s=30)
        axis.plot([0, 100], [0, 100], "r--", linewidth=2, label="y=x (perfect agreement)")
        axis.set_xlabel(f"Quality estimated by {label} (%)")
        axis.set_ylabel("Combined estimated quality (%)")
        axis.set_title(f"{label} vs combined method")
        axis.legend()
        axis.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=axis, label="Number of compressions")

    axes[1, 1].axis("off")
    plt.tight_layout()
    plt.show()

    print("\n=== Correlation between quality-estimation methods ===")
    print(results_df[quality_columns].corr())
    print("\n=== Standard deviation by number of compressions ===")
    print(results_df.groupby("compression_count")[quality_columns].std())
